# 🛡️ Deepfake Detector — Backend (GPU)
Run this notebook on Google Colab with **GPU runtime** to get fast inference.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells (Ctrl+F9)
3. Copy the ngrok URL printed at the end
4. Set it as `VITE_API_URL` in your frontend `.env`

In [ ]:
# Cell 1: Clone your repo
!git clone https://github.com/Tanish024/Deep.git
%cd Deep/backend

In [ ]:
# Cell 2: Install dependencies (GPU version of PyTorch auto-installed on Colab)
!pip install fastapi uvicorn transformers huggingface_hub \
    opencv-python-headless python-multipart pydantic Pillow numpy pyngrok

In [ ]:
# Cell 3: Test GPU availability
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# Cell 4: Patch detector.py to use GPU if available
import os
os.environ['HF_HOME'] = './models'
os.environ['FAKE_THRESHOLD'] = '0.5'

# Read and patch detector to use cuda
detector_path = 'app/detector.py'
with open(detector_path, 'r') as f:
    code = f.read()

code = code.replace(
    '_DEVICE = "cpu"',
    '_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"'
)
code = 'import torch\n' + code  # ensure torch is imported before the device line

with open(detector_path, 'w') as f:
    f.write(code)

print('✅ Patched detector.py for GPU')

In [ ]:
# Cell 5: Quick test — load model and run inference
from app.detector import classify_frame
from PIL import Image
import numpy as np

img = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))
score = classify_frame(img)
print(f'✅ Model works! Test score: {score:.4f}')

In [ ]:
# Cell 6: Setup ngrok tunnel (exposes your Colab backend to the internet)
# Get a free auth token at: https://dashboard.ngrok.com/signup
# Then paste it below:

NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # <-- PASTE YOUR TOKEN

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8000)
print(f'\n🌐 Your backend is live at:')
print(f'   {public_url}')
print(f'\n📋 Copy this URL and set it as VITE_API_URL in your frontend .env')
print(f'   VITE_API_URL={public_url}')

In [ ]:
# Cell 7: Start the FastAPI server (runs forever — keep this cell running!)
# Using ! to run as subprocess — avoids Colab's event loop conflict
!HF_HOME=./models FAKE_THRESHOLD=0.5 uvicorn app.main:app --host 0.0.0.0 --port 8000